In [197]:
import numpy as np
import matplotlib.pyplot as plt
import cupy as cp
import time
from memory_profiler import memory_usage

In [198]:
SEED = 42
N_PARTICLES = 3000
BOX_LENGHT = 1
SPACE_DIM = 3

\begin{equation*}
U_{c} = \frac{1}{2}\sum_{ij}\frac{q_iq_j}{||r_i - r_j||}
\end{equation*}

\begin{equation*}
D_{ij}^2 = \bold{r}_i \cdot \bold{r}_i + \bold{r}_j \cdot \bold{r}_j - 2 \bold{r}_i \cdot \bold{r}_j
\end{equation*}

\begin{equation*}
U_{c} = \frac{1}{2} \bold{q} \cdot \Lambda\bold{q}
\end{equation*}

In [199]:
particle_type = np.dtype([
    ('position', np.float64, (SPACE_DIM,)),
    ('velocity', np.float64, (SPACE_DIM,)),
    ('mass', np.float64),
    ('charge', np.float64)
])

In [200]:
def compute_distance_square_matrix(positions : np.array) -> np.array:
    
    r2 = np.sum(positions*positions, axis=1)
    D2 = r2[:,None] + r2[None,:] - 2*(positions @ positions.T)
    np.fill_diagonal(D2, np.inf)
    
    return D2

In [201]:
def compute_coulomb_energy(charges : np.array, D2 : np.array) -> np.float64:

    energy = 0.5 * np.sum((charges[:,None] * charges[None,:]) / D2)
    
    return energy

In [202]:
def compute_lennar_jones_energy(D2 : np.array) -> np.float64:
    
    D8 = np.empty_like(D2)
    np.power(D2, 4, out=D8)
    energy = np.sum(1 / D8)

    return energy

In [203]:
def compute_potential_energy(particles : np.ndarray) -> np.float64:
    
    D2 = compute_distance_square_matrix(particles['position'])
    
    energy_coulomb = compute_coulomb_energy(particles['charge'], D2)
    #energy_lennar = compute_lennar_jones_energy(D2)
    
    return energy_coulomb    

In [206]:
N = [100, 1000, 1500, 2000, 3000, 4000,10000]


for N_PARTICLES in N:
    particles = np.zeros(N_PARTICLES, dtype=particle_type)

    particles['position'] = np.random.uniform(low=0,high=1,size=(N_PARTICLES,SPACE_DIM))
    particles['velocity'] = np.random.uniform(low=0,high=1,size=(N_PARTICLES,SPACE_DIM))
    particles['mass'] = 1
    particles['charge'] = np.sign(np.random.uniform(-1, 1, size=N_PARTICLES))
    
    
    mem_usage = memory_usage(lambda: compute_potential_energy(particles=particles))
    
    delta_time= timeit.timeit(lambda: compute_potential_energy(particles=particles), number=1)
    
    print(f"N={N_PARTICLES} particelle")
    print(f"Tempo: {delta_time:.6f} secondi")
    print(f"Memoria massima: {max(mem_usage)} MiB")

N=100 particelle
Tempo: 0.000183 secondi
Memoria massima: 161.4140625 MiB
N=1000 particelle
Tempo: 0.014747 secondi
Memoria massima: 176.0625 MiB
N=1500 particelle
Tempo: 0.029145 secondi
Memoria massima: 195.45703125 MiB
N=2000 particelle
Tempo: 0.088668 secondi
Memoria massima: 222.05078125 MiB
N=3000 particelle
Tempo: 0.206429 secondi
Memoria massima: 298.73046875 MiB
N=4000 particelle
Tempo: 0.352448 secondi
Memoria massima: 404.49609375 MiB
N=10000 particelle
Tempo: 1.913010 secondi
Memoria massima: 1687.44140625 MiB
